# Notebook 06 — Evaluation of Enhanced 3D ResUNet

This notebook evaluates the persisted Enhanced 3D ResUNet multi-task model produced by Notebook 05.

**Current scope:** RibFrac128 development/smoke-test experiment. These results are not final thesis performance.

The notebook loads the persistent patch dataset and training-results dataset, reproduces the deterministic validation split used in Notebook 05, evaluates segmentation and five-class classification, and saves evaluation artifacts for Notebook 07.


In [ ]:
# ============================================================
# CELL 1 — IMPORTS AND REPRODUCIBILITY
# ============================================================

import os
import json
import random
from pathlib import Path

import numpy as np
import pandas as pd
import tensorflow as tf
import matplotlib.pyplot as plt

from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix,
    roc_auc_score,
)

SEED = 42

np.random.seed(SEED)
tf.random.set_seed(SEED)
random.seed(SEED)

print("Libraries imported successfully.")
print("TensorFlow version:", tf.__version__)
print("Random seed:", SEED)


In [ ]:
# ============================================================
# CELL 2 — GPU CHECK
# ============================================================

gpus = tf.config.list_physical_devices("GPU")

if gpus:
    print("GPU detected:")
    for gpu in gpus:
        print(gpu)
else:
    print("WARNING: No GPU detected.")


## Persistent Kaggle datasets

- Patch dataset: `waboke/ribfrac-patch-dataset`
- Training results: `waboke/ribfrac-resunet-training-results`

The second dataset contains the best checkpoint, final model, training history, validation metrics, and experiment configuration produced by Notebook 05.


In [ ]:
# ============================================================
# CELL 3 — DEFINE PERSISTENT PATHS
# ============================================================

PATCH_DATASET_ROOT = Path(
    "/kaggle/input/datasets/waboke/ribfrac-patch-dataset"
)

TRAINING_RESULTS_ROOT = Path(
    "/kaggle/input/datasets/waboke/ribfrac-resunet-training-results"
)

print("Patch dataset:")
print(PATCH_DATASET_ROOT)

print()
print("Training results:")
print(TRAINING_RESULTS_ROOT)


In [ ]:
# ============================================================
# CELL 4 — VERIFY DATASETS
# ============================================================

if not PATCH_DATASET_ROOT.exists():
    raise FileNotFoundError(
        f"Patch dataset not found:\n{PATCH_DATASET_ROOT}"
    )

if not TRAINING_RESULTS_ROOT.exists():
    raise FileNotFoundError(
        f"Training-results dataset not found:\n{TRAINING_RESULTS_ROOT}"
    )

print("Both persistent datasets are accessible.")


In [ ]:
# ============================================================
# CELL 5 — DISPLAY TRAINING ARTIFACTS
# ============================================================

for path in sorted(TRAINING_RESULTS_ROOT.rglob("*")):
    if path.is_file():
        size_mb = path.stat().st_size / (1024 ** 2)
        print(
            f"{path.relative_to(TRAINING_RESULTS_ROOT)}"
            f"    {size_mb:.2f} MB"
        )


In [ ]:
# ============================================================
# CELL 6 — DEVELOPMENT DATA PATHS
# ============================================================

DEV_ROOT = PATCH_DATASET_ROOT / "RibFrac128"

X_PATH = DEV_ROOT / "X.npy"
Y_SEG_PATH = DEV_ROOT / "Y_seg.npy"
Y_CLASS_PATH = DEV_ROOT / "Y_class.npy"

CONFIG_PATH = PATCH_DATASET_ROOT / "patch_extraction_config.json"

print("X:", X_PATH)
print("Y_seg:", Y_SEG_PATH)
print("Y_class:", Y_CLASS_PATH)
print("Patch config:", CONFIG_PATH)


In [ ]:
# ============================================================
# CELL 7 — LOAD AND VALIDATE DEVELOPMENT DATA
# ============================================================

for path in [X_PATH, Y_SEG_PATH, Y_CLASS_PATH]:
    if not path.is_file():
        raise FileNotFoundError(f"Required file not found:\n{path}")

X = np.load(X_PATH).astype(np.float32)
Y_seg = np.load(Y_SEG_PATH).astype(np.float32)
Y_class = np.load(Y_CLASS_PATH).astype(np.int32)

assert X.ndim == 5
assert Y_seg.ndim == 5
assert Y_class.ndim == 1
assert X.shape[1:] == (64, 64, 64, 1)
assert Y_seg.shape[1:] == (64, 64, 64, 1)
assert X.shape[0] == Y_seg.shape[0] == Y_class.shape[0]
assert np.isfinite(X).all()
assert np.isfinite(Y_seg).all()
assert np.isfinite(Y_class).all()

print("X shape:", X.shape)
print("Y_seg shape:", Y_seg.shape)
print("Y_class shape:", Y_class.shape)
print("Development dataset validation: PASS")


In [ ]:
# ============================================================
# CELL 8 — REPRODUCE NOTEBOOK 05 VALIDATION SPLIT
# ============================================================

# This exactly reproduces the deterministic 80/20 split used
# in Notebook 05 for the current development experiment.

NUM_SAMPLES = len(X)

indices = np.arange(NUM_SAMPLES)

rng = np.random.default_rng(SEED)
rng.shuffle(indices)

split_index = int(0.8 * NUM_SAMPLES)

train_indices = indices[:split_index]
val_indices = indices[split_index:]

X_val = X[val_indices]
Y_seg_val = Y_seg[val_indices]
Y_class_val = Y_class[val_indices]

print("Total patches:", NUM_SAMPLES)
print("Validation patches:", len(X_val))

unique_classes, counts = np.unique(
    Y_class_val,
    return_counts=True
)

print()
print("Validation class distribution:")
for class_id, count in zip(unique_classes, counts):
    print(f"Class {class_id}: {count}")


In [ ]:
# ============================================================
# CELL 9 — CUSTOM METRICS FOR MODEL LOADING
# ============================================================

def dice_coefficient(y_true, y_pred):
    smooth = 1e-6

    y_true = tf.cast(y_true, tf.float32)
    y_pred = tf.cast(y_pred, tf.float32)
    y_pred = tf.cast(y_pred > 0.5, tf.float32)

    intersection = tf.reduce_sum(
        y_true * y_pred, axis=[1, 2, 3, 4]
    )

    denominator = (
        tf.reduce_sum(y_true, axis=[1, 2, 3, 4])
        + tf.reduce_sum(y_pred, axis=[1, 2, 3, 4])
    )

    return tf.reduce_mean(
        (2.0 * intersection + smooth)
        / (denominator + smooth)
    )


def iou_coefficient(y_true, y_pred):
    smooth = 1e-6

    y_true = tf.cast(y_true, tf.float32)
    y_pred = tf.cast(y_pred, tf.float32)
    y_pred = tf.cast(y_pred > 0.5, tf.float32)

    intersection = tf.reduce_sum(
        y_true * y_pred, axis=[1, 2, 3, 4]
    )

    union = (
        tf.reduce_sum(y_true, axis=[1, 2, 3, 4])
        + tf.reduce_sum(y_pred, axis=[1, 2, 3, 4])
        - intersection
    )

    return tf.reduce_mean(
        (intersection + smooth) / (union + smooth)
    )

print("Custom metrics defined.")


In [ ]:
# ============================================================
# CELL 10 — LOAD EXPERIMENT CONFIGURATION
# ============================================================

EXPERIMENT_CONFIG_PATH = (
    TRAINING_RESULTS_ROOT / "experiment_config.json"
)

if not EXPERIMENT_CONFIG_PATH.is_file():
    raise FileNotFoundError(
        f"Experiment configuration not found:\n"
        f"{EXPERIMENT_CONFIG_PATH}"
    )

with open(EXPERIMENT_CONFIG_PATH, "r") as f:
    experiment_config = json.load(f)

print(json.dumps(experiment_config, indent=4))


In [ ]:
# ============================================================
# CELL 11 — LOAD PERSISTED BEST MODEL
# ============================================================

BEST_MODEL_PATH = (
    TRAINING_RESULTS_ROOT
    / "checkpoints"
    / "best_enhanced_3d_resunet.keras"
)

if not BEST_MODEL_PATH.is_file():
    raise FileNotFoundError(
        f"Best model not found:\n{BEST_MODEL_PATH}"
    )

best_model = tf.keras.models.load_model(
    BEST_MODEL_PATH,
    custom_objects={
        "dice_coefficient": dice_coefficient,
        "iou_coefficient": iou_coefficient,
    }
)

print("Best model loaded successfully.")
print("Input shape:", best_model.input_shape)
print("Output shapes:", best_model.output_shape)


In [ ]:
# ============================================================
# CELL 12 — MODEL ARCHITECTURE VALIDATION
# ============================================================

assert best_model.input_shape == (
    None, 64, 64, 64, 1
)

assert isinstance(best_model.output_shape, list)
assert len(best_model.output_shape) == 2

print("Model architecture validation: PASS")


In [ ]:
# ============================================================
# CELL 13 — KERAS VALIDATION EVALUATION
# ============================================================

evaluation_results = best_model.evaluate(
    X_val,
    {
        "segmentation_output": Y_seg_val,
        "classification_output": Y_class_val,
    },
    batch_size=1,
    verbose=1,
    return_dict=True,
)

print()
print("Validation results:")
for metric, value in evaluation_results.items():
    print(f"{metric}: {value:.6f}")


In [ ]:
# ============================================================
# CELL 14 — INFERENCE
# ============================================================

predictions = best_model.predict(
    X_val,
    batch_size=1,
    verbose=1
)

# The loaded Keras model can expose outputs as an ordered list.
# Identify outputs by dimensionality rather than assuming order.

if isinstance(predictions, dict):
    classification_prob = np.asarray(
        predictions["classification_output"]
    )
    segmentation_prob = np.asarray(
        predictions["segmentation_output"]
    )
else:
    prediction_list = list(predictions)

    classification_candidates = [
        p for p in prediction_list if p.ndim == 2
    ]

    segmentation_candidates = [
        p for p in prediction_list if p.ndim == 5
    ]

    if len(classification_candidates) != 1:
        raise RuntimeError(
            "Could not uniquely identify classification output."
        )

    if len(segmentation_candidates) != 1:
        raise RuntimeError(
            "Could not uniquely identify segmentation output."
        )

    classification_prob = np.asarray(
        classification_candidates[0]
    )
    segmentation_prob = np.asarray(
        segmentation_candidates[0]
    )

print("Classification predictions:",
      classification_prob.shape)

print("Segmentation predictions:",
      segmentation_prob.shape)


In [ ]:
# ============================================================
# CELL 15 — CLASSIFICATION METRICS
# ============================================================

class_names = [
    "Background",
    "Displaced",
    "Non-displaced",
    "Buckle",
    "Segmental",
]

y_true_class = Y_class_val.astype(np.int32)

y_pred_class = np.argmax(
    classification_prob,
    axis=1
).astype(np.int32)

classification_metrics = {
    "accuracy": accuracy_score(
        y_true_class, y_pred_class
    ),
    "balanced_accuracy": balanced_accuracy_score(
        y_true_class, y_pred_class
    ),
    "macro_precision": precision_score(
        y_true_class, y_pred_class,
        average="macro", zero_division=0
    ),
    "macro_recall": recall_score(
        y_true_class, y_pred_class,
        average="macro", zero_division=0
    ),
    "macro_f1": f1_score(
        y_true_class, y_pred_class,
        average="macro", zero_division=0
    ),
    "weighted_precision": precision_score(
        y_true_class, y_pred_class,
        average="weighted", zero_division=0
    ),
    "weighted_recall": recall_score(
        y_true_class, y_pred_class,
        average="weighted", zero_division=0
    ),
    "weighted_f1": f1_score(
        y_true_class, y_pred_class,
        average="weighted", zero_division=0
    ),
}

print("Classification metrics:")
for metric, value in classification_metrics.items():
    print(f"{metric}: {value:.6f}")


In [ ]:
# ============================================================
# CELL 16 — CLASSIFICATION REPORT
# ============================================================

present_class_ids = sorted(
    np.unique(
        np.concatenate([y_true_class, y_pred_class])
    ).tolist()
)

report = classification_report(
    y_true_class,
    y_pred_class,
    labels=present_class_ids,
    target_names=[
        class_names[i] for i in present_class_ids
    ],
    zero_division=0,
    output_dict=True,
)

classification_report_df = pd.DataFrame(report).T

print(classification_report_df)


In [ ]:
# ============================================================
# CELL 17 — CONFUSION MATRIX
# ============================================================

cm = confusion_matrix(
    y_true_class,
    y_pred_class,
    labels=list(range(len(class_names)))
)

cm_df = pd.DataFrame(
    cm,
    index=class_names,
    columns=class_names
)

print(cm_df)


In [ ]:
# ============================================================
# CELL 18 — PLOT CONFUSION MATRIX
# ============================================================

plt.figure(figsize=(8, 7))
plt.imshow(cm)

plt.title(
    "RibFrac128 Development Validation Confusion Matrix"
)
plt.xlabel("Predicted Class")
plt.ylabel("True Class")

plt.xticks(
    range(len(class_names)),
    class_names,
    rotation=45,
    ha="right"
)

plt.yticks(
    range(len(class_names)),
    class_names
)

for i in range(cm.shape[0]):
    for j in range(cm.shape[1]):
        plt.text(
            j, i, cm[i, j],
            ha="center",
            va="center"
        )

plt.colorbar()
plt.tight_layout()
plt.show()


In [ ]:
# ============================================================
# CELL 19 — MULTICLASS ROC-AUC
# ============================================================

try:
    roc_auc = roc_auc_score(
        y_true_class,
        classification_prob,
        multi_class="ovr",
        average="macro",
        labels=list(range(len(class_names))),
    )

    print(
        "Macro ROC-AUC (OvR):",
        f"{roc_auc:.6f}"
    )

except ValueError as error:
    roc_auc = None

    print(
        "Macro ROC-AUC is not mathematically valid "
        "for this validation subset."
    )
    print("Reason:", error)


In [ ]:
# ============================================================
# CELL 20 — SEGMENTATION METRICS
# ============================================================

y_true_seg = (
    Y_seg_val >= 0.5
).astype(np.float32)

y_pred_seg = (
    segmentation_prob >= 0.5
).astype(np.float32)

def segmentation_metrics_per_patch(
    y_true,
    y_pred,
    smooth=1e-6
):
    yt = y_true.reshape(y_true.shape[0], -1)
    yp = y_pred.reshape(y_pred.shape[0], -1)

    intersection = np.sum(yt * yp, axis=1)
    true_sum = np.sum(yt, axis=1)
    pred_sum = np.sum(yp, axis=1)

    union = true_sum + pred_sum - intersection

    dice = (
        2.0 * intersection + smooth
    ) / (
        true_sum + pred_sum + smooth
    )

    iou = (
        intersection + smooth
    ) / (
        union + smooth
    )

    precision = (
        intersection + smooth
    ) / (
        pred_sum + smooth
    )

    recall = (
        intersection + smooth
    ) / (
        true_sum + smooth
    )

    return dice, iou, precision, recall


dice_values, iou_values, seg_precision, seg_recall = (
    segmentation_metrics_per_patch(
        y_true_seg,
        y_pred_seg
    )
)

segmentation_metrics = {
    "mean_dice": float(np.mean(dice_values)),
    "std_dice": float(np.std(dice_values)),
    "mean_iou": float(np.mean(iou_values)),
    "std_iou": float(np.std(iou_values)),
    "mean_precision": float(np.mean(seg_precision)),
    "mean_recall": float(np.mean(seg_recall)),
}

for metric, value in segmentation_metrics.items():
    print(f"{metric}: {value:.6f}")


In [ ]:
# ============================================================
# CELL 21 — QUALITATIVE SEGMENTATION CHECK
# ============================================================

num_examples = min(4, len(X_val))

for idx in range(num_examples):

    center_z = X_val[idx].shape[2] // 2

    ct_slice = X_val[idx, :, :, center_z, 0]
    true_slice = y_true_seg[idx, :, :, center_z, 0]
    pred_slice = y_pred_seg[idx, :, :, center_z, 0]

    fig = plt.figure(figsize=(12, 4))

    ax1 = fig.add_subplot(1, 3, 1)
    ax1.imshow(ct_slice, cmap="gray")
    ax1.set_title(f"CT — patch {idx}")
    ax1.axis("off")

    ax2 = fig.add_subplot(1, 3, 2)
    ax2.imshow(true_slice)
    ax2.set_title("Ground Truth")
    ax2.axis("off")

    ax3 = fig.add_subplot(1, 3, 3)
    ax3.imshow(pred_slice)
    ax3.set_title("Prediction")
    ax3.axis("off")

    plt.tight_layout()
    plt.show()


In [ ]:
# ============================================================
# CELL 22 — CREATE EVALUATION OUTPUT DIRECTORY
# ============================================================

OUTPUT_ROOT = Path(
    "/kaggle/working/evaluation_results"
)

OUTPUT_ROOT.mkdir(
    parents=True,
    exist_ok=True
)

print("Evaluation output directory:")
print(OUTPUT_ROOT)


In [ ]:
# ============================================================
# CELL 23 — SAVE EVALUATION RESULTS
# ============================================================

results = {
    "evaluation_scope": (
        "RibFrac128 development/smoke-test validation subset"
    ),
    "seed": SEED,
    "num_total_patches": int(NUM_SAMPLES),
    "num_validation_patches": int(len(X_val)),
    "classification_metrics": classification_metrics,
    "segmentation_metrics": segmentation_metrics,
    "roc_auc_macro_ovr": (
        None if roc_auc is None else float(roc_auc)
    ),
}

with open(
    OUTPUT_ROOT / "evaluation_results.json",
    "w"
) as f:
    json.dump(results, f, indent=4)

classification_report_df.to_csv(
    OUTPUT_ROOT / "classification_report.csv"
)

cm_df.to_csv(
    OUTPUT_ROOT / "confusion_matrix.csv"
)

pd.DataFrame({
    "Metric": ["Dice", "IoU", "Precision", "Recall"],
    "Mean": [
        np.mean(dice_values),
        np.mean(iou_values),
        np.mean(seg_precision),
        np.mean(seg_recall),
    ],
    "Std": [
        np.std(dice_values),
        np.std(iou_values),
        np.std(seg_precision),
        np.std(seg_recall),
    ],
}).to_csv(
    OUTPUT_ROOT / "segmentation_metrics.csv",
    index=False
)

evaluation_config = {
    "notebook": "06_Evaluation_Enhanced_3D_ResUNet_RibFrac",
    "evaluation_scope": (
        "RibFrac128 development/smoke-test validation subset"
    ),
    "patch_dataset": "waboke/ribfrac-patch-dataset",
    "training_results_dataset": (
        "waboke/ribfrac-resunet-training-results"
    ),
    "model_file": (
        "checkpoints/best_enhanced_3d_resunet.keras"
    ),
    "seed": SEED,
    "validation_split": 0.20,
    "segmentation_threshold": 0.5,
    "classification_method": "argmax",
}

with open(
    OUTPUT_ROOT / "evaluation_config.json",
    "w"
) as f:
    json.dump(evaluation_config, f, indent=4)

print("Evaluation artifacts saved.")


In [ ]:
# ============================================================
# CELL 24 — FINAL SUMMARY
# ============================================================

print("=" * 60)
print("NOTEBOOK 06 — DEVELOPMENT EVALUATION COMPLETED")
print("=" * 60)

print()
print("Validation patches:", len(X_val))

print()
print("Segmentation:")
print(
    f"Mean Dice: {segmentation_metrics['mean_dice']:.6f}"
)
print(
    f"Mean IoU:  {segmentation_metrics['mean_iou']:.6f}"
)

print()
print("Classification:")
print(
    f"Accuracy: {classification_metrics['accuracy']:.6f}"
)
print(
    f"Macro F1: {classification_metrics['macro_f1']:.6f}"
)

if roc_auc is not None:
    print(
        f"Macro ROC-AUC: {roc_auc:.6f}"
    )
else:
    print("Macro ROC-AUC: not available for this subset.")

print()
print("Saved to:")
print(OUTPUT_ROOT)

print()
print(
    "NOTE: These are development/smoke-test results, "
    "not final thesis results."
)

print("=" * 60)
